In [ ]:
import torch
import torch.nn as nn

class MiniModel(nn.Module):
    def forward(self, x):
        y = torch.relu(x)
        z = y + 1.0
        return z

model = MiniModel().eval()
x = torch.tensor([[-1.0, 0.5, 2.0]], dtype=torch.float32)

torch.onnx.export(
    model,
    x,
    "work/mini.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=13
)

print("exported: work/mini.onnx")

In [ ]:
import onnx
from onnx import helper

model = onnx.load("work/mini.onnx")
graph = model.graph

# 找到 Add 节点，把它替换成自定义节点
new_nodes = []
for node in graph.node:
    if node.op_type == "Add":
        # 只保留第一个输入（Relu输出），把常量1.0改成属性alpha=2.0
        custom_node = helper.make_node(
            "MyScale",
            inputs=[node.input[0]],
            outputs=list(node.output),
            domain="my.custom",
            alpha=2.0
        )
        new_nodes.append(custom_node)
    else:
        new_nodes.append(node)

del graph.node[:]
graph.node.extend(new_nodes)

# 重要：注册自定义 domain 的 opset
model.opset_import.extend([helper.make_opsetid("my.custom", 1)])

onnx.save(model, "work/mini_custom.onnx")
print("saved: work/mini_custom.onnx")